# NB-05-02 — Variable Elimination

Exact inference by factor algebra on the alarm network, checked against enumeration and against a
brute-force sum over the 32-entry joint table. Nothing is hardcoded: every posterior printed here is
computed from the conditional probability tables.

Uses `code/variable_elimination.py` (generic factors) and `code/alarm_network.py` (the network, its
enumeration reference and its tests).

In [1]:
import pathlib, sys

CODE = None
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / "code" / "variable_elimination.py").exists():
        CODE = base / "code"
        break
if CODE is None:
    CODE = pathlib.Path.cwd().parent / "code"
sys.path.insert(0, str(CODE))

import numpy as np
import alarm_network as net
import variable_elimination as ve
print("numpy", np.__version__, "| code:", CODE)

numpy 2.5.3 | code: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M05-uncertainty-bayesian-reasoning-and-fuzzy-logic\code


## 1. A factor is a table plus its variables

`P(B)` and the evidence-restricted `P(M | A)` as factors.

In [2]:
f_b = ve.Factor.from_array(["B"], [0.001, 0.999])
print("B factor:", f_b.variables, f_b.table, "sum =", sum(f_b.table.values()))

f_m = ve.Factor.from_array(["M", "A"], [[0.70, 0.30], [0.01, 0.99]])
print("M|A factor:", f_m.variables)
print("  P(m|a) =", f_m.table[(True, True)], " P(m|-a) =", f_m.table[(True, False)])
print("restrict M=True ->", f_m.restrict({"M": True}).variables, f_m.restrict({"M": True}).table)

B factor: ('B',) {(True,): 0.001, (False,): 0.999} sum = 1.0
M|A factor: ('M', 'A')
  P(m|a) = 0.7  P(m|-a) = 0.3
restrict M=True -> ('A',) {(True,): 0.7, (False,): 0.3}


## 2. Factor operations, checked by hand

Multiply, sum out, and normalise on a two-row table where the arithmetic is obvious.

In [3]:
a = ve.Factor.from_array(["X"], [0.2, 0.8])
b = ve.Factor.from_array(["X"], [0.5, 0.1])
prod = a.multiply(b)
print("product:", prod.table)
print("sum out X:", prod.sum_out("X").table, " (single-factor sums are not probabilities)")
normed, total = prod.normalize()
print("normalised:", normed.table, "total =", total)

p = ve.Factor.from_array(["X"], [0.2, 0.8]).multiply(ve.Factor.from_array(["Y"], [0.3, 0.7]))
print("independent product:", p.table, "checks P(T,T)=0.06")

product: {(True,): 0.1, (False,): 0.08000000000000002}
sum out X: {(): 0.18000000000000002}  (single-factor sums are not probabilities)
normalised: {(True,): 0.5555555555555555, (False,): 0.4444444444444445} total = 0.18000000000000002
independent product: {(True, True): 0.06, (True, False): 0.13999999999999999, (False, True): 0.24, (False, False): 0.5599999999999999} checks P(T,T)=0.06


## 3. The elimination trace for P(B | j, m)

Order E then A. Each step shows the factor that was summed out and what replaced it.

In [4]:
trace = []
p_ve, p_evidence = ve.variable_elimination(
    ve.alarm_factors(), "B", {"J": True, "M": True}, order=["E", "A"], trace=trace)
for step in trace:
    print(f"eliminate {step['eliminated']}: {step['before'].variables} -> {step['after'].variables}")
    for k, v in sorted(step["after"].table.items()):
        print("   ", k, round(v, 6))
print("posterior:", round(p_ve, 4), " P(j,m):", round(p_evidence, 8))

eliminate E: ('E', 'A', 'B') -> ('A', 'B')
    (False, False) 0.998422
    (False, True) 0.05998
    (True, False) 0.001578
    (True, True) 0.94002
eliminate A: ('A', 'B') -> ('B',)
    (False,) 0.001493
    (True,) 0.592243
posterior: 0.2842  P(j,m): 0.0020841


## 4. Eliminate in the other order

Same answer, different intermediate factors: ordering is a cost decision, not a correctness one.

In [5]:
trace2 = []
p_ve2, _ = ve.variable_elimination(
    ve.alarm_factors(), "B", {"J": True, "M": True}, order=["A", "E"], trace=trace2)
for step in trace2:
    print(f"eliminate {step['eliminated']}: {step['before'].variables} -> {step['after'].variables}",
          "| largest table:", len(step["before"].table))
print("posterior:", round(p_ve2, 6), "| identical:", abs(p_ve - p_ve2) < 1e-12)

eliminate A: ('A', 'B', 'E') -> ('B', 'E') | largest table: 8
eliminate E: ('E', 'B') -> ('B',) | largest table: 4
posterior: 0.284172 | identical: True


## 5. VE versus enumeration versus brute force

Three independent routes to the same number, for nine evidence combinations.

In [6]:
table = net.joint_table()
names = net.NODES

def brute_force(ev):
    den = sum(v for k, v in table.items()
              if all(k[names.index(n)] == x for n, x in ev.items()))
    num = sum(v for k, v in table.items()
              if k[0] is True and all(k[names.index(n)] == x for n, x in ev.items()))
    return num / den

print(f"{'evidence':24s} {'VE':>8s} {'enum':>8s} {'brute':>8s} {'max diff':>10s}")
worst = 0.0
for label, ev in ve.EVIDENCE_CASES:
    fixed = {n for n in ("E", "A") if n in ev}
    order = [v for v in ("E", "A") if v not in fixed]
    p_v, _ = ve.variable_elimination(ve.alarm_factors(), "B", ev, order=order)
    p_e, _ = net.enumerate_posterior("B", ev)
    p_bf = brute_force(ev)
    diff = max(abs(p_v - p_e), abs(p_v - p_bf))
    worst = max(worst, diff)
    print(f"{label:24s} {p_v:8.4f} {p_e:8.4f} {p_bf:8.4f} {diff:10.1e}")
print("worst absolute difference:", worst)
assert worst < 1e-12

evidence                       VE     enum    brute   max diff
no evidence                0.0010   0.0010   0.0010    0.0e+00
John calls                 0.0163   0.0163   0.0163    6.9e-18
Mary calls                 0.0561   0.0561   0.0561    0.0e+00
both call                  0.2842   0.2842   0.2842    1.1e-16
John only                  0.0051   0.0051   0.0051    2.6e-18
Mary only                  0.0069   0.0069   0.0069    2.6e-18
neither calls              0.0001   0.0001   0.0001    0.0e+00
earthquake                 0.0010   0.0010   0.0010    0.0e+00
earthquake, both call      0.0033   0.0033   0.0033    4.3e-19
worst absolute difference: 1.1102230246251565e-16


## 6. Tests

Factor algebra, agreement with both reference implementations, order independence, and rejection of
invalid input.

In [7]:
print("alarm_network self-tests:", net.run_tests())
print("variable_elimination self-tests:", ve.run_tests())

order_independent = all(
    abs(ve.variable_elimination(ve.alarm_factors(), "B", {"J": True, "M": True}, order=o)[0] - 0.2841718) < 1e-6
    for o in (["E", "A"], ["A", "E"]))
print("PASS  order independent" if order_independent else "FAIL  order independent")

for bad in ({"B": True}, {"X": True}, {"J": 1}):
    try:
        ve.variable_elimination(ve.alarm_factors(), "B", bad, order=["E", "A"])
    except ValueError as exc:
        print("  rejected", bad, "->", str(exc)[:60])
    else:
        print("  FAIL: accepted", bad)

alarm_network self-tests: True
variable_elimination self-tests: True
PASS  order independent
  rejected {'B': True} -> query variable appears in the evidence
  rejected {'X': True} -> evidence names unknown variables: ['X']
  rejected {'J': 1} -> evidence values must be True or False, got [1]


## 7. Cost: why ordering matters

Count the entries in every intermediate factor for both orders. On this network the difference is
small; the same arithmetic on 20 nodes is not.

In [8]:
def cost(order):
    trace = []
    ve.variable_elimination(ve.alarm_factors(), "B", {"J": True, "M": True}, order=order, trace=trace)
    return [len(s["before"].table) for s in trace], sum(len(s["before"].table) for s in trace)

for order in (["E", "A"], ["A", "E"]):
    sizes, total = cost(order)
    print(f"order {order}: intermediate sizes {sizes}, total entries {total}")

print("\nenumeration: 32 joint evaluations, 2 x 16 additions per query")
print("VE: 2 eliminations, largest intermediate 8 entries")

order ['E', 'A']: intermediate sizes [8, 4], total entries 12
order ['A', 'E']: intermediate sizes [8, 4], total entries 12

enumeration: 32 joint evaluations, 2 x 16 additions per query
VE: 2 eliminations, largest intermediate 8 entries


## What you should take away

- A factor is a table with a variable list; restrict, multiply, sum out, normalise are all it takes.
- VE is exact: it is the same sum, reordered so each variable is discarded once.
- The trace above reproduces L05-06 section 4: g1(A,B) then g2(B)=0.5922426 / 0.0014934, then normalise
  to 0.2842.
- Ordering changes cost, never the answer.
- Three independent implementations agreeing to 1e-12 is the standard this module expects.

Next: [LAB-05-01](../labs/lab-05-01-bayesian-and-fuzzy-decision-system.md).